In [1]:
import os, sys, subprocess, numpy as np
import json, time, math, random
from collections import defaultdict, Counter
from PIL import Image, ImageDraw, ImageFont
import torch, torch.nn as nn, pywt, warnings
import unicodedata
warnings.filterwarnings('ignore')

subprocess.run(['pip', 'install', 'transformers',
                'sentencepiece', 'sacrebleu',
                'evaluate', 'openpyxl', '-q'])

# ── Paths ─────────────────────────────────────────────────────
KIN_PATH  = ('/kaggle/input/datasets/sumitsharma2005'
             '/kinnauri-hindi-dataset')
CLL_PATH  = ('/kaggle/input/datasets/sumitsharma2005'
             '/hindi-en-scalograms/CLL-STR')
FONT_PATH = '/kaggle/working/NotoSansDevanagari-Regular.ttf'
WORK_DIR  = '/kaggle/working'
CLEAN_DIR = f"{WORK_DIR}/kinnauri_hindi_cleaned"

os.makedirs(CLEAN_DIR, exist_ok=True)
sys.path.insert(0, CLL_PATH)

# Exact filenames with spaces
HINDI_FILE   = f"{KIN_PATH}/Parallel_data_Hi (1).txt"
KINNAURI_FILE= f"{KIN_PATH}/Parallel_data_KP (1).txt"

# ── Load and clean ────────────────────────────────────────────
def clean_lines(lines):
    out = []
    for l in lines:
        l = unicodedata.normalize('NFC', l.strip())
        if l:
            out.append(l)
    return out

with open(KINNAURI_FILE, encoding='utf-8',
          errors='ignore') as f:
    kc = clean_lines(f.readlines())
with open(HINDI_FILE, encoding='utf-8',
          errors='ignore') as f:
    hc = clean_lines(f.readlines())

print(f"Kinnauri: {len(kc):,}")
print(f"Hindi:    {len(hc):,}")
assert len(kc) == len(hc), \
    f"Mismatch: {len(kc)} vs {len(hc)}"
print("✅ Files perfectly aligned")

# ── Filter sentences ≥ 300 chars ─────────────────────────────
pairs = []
skipped = 0
for k, h in zip(kc, hc):
    if len(k) < 300 and len(h) < 300:
        pairs.append((k, h))
    else:
        skipped += 1
print(f"After length filter: {len(pairs):,} pairs "
      f"(skipped {skipped})")

# ── Shuffle + split ───────────────────────────────────────────
random.seed(42)
random.shuffle(pairs)

n_total = len(pairs)
n_test  = 500
n_dev   = 500
n_train = n_total - n_test - n_dev

train_pairs = pairs[:n_train]
dev_pairs   = pairs[n_train:n_train+n_dev]
test_pairs  = pairs[n_train+n_dev:]

print(f"\nSplit:")
print(f"  Train: {len(train_pairs):,}")
print(f"  Dev:   {len(dev_pairs):,}")
print(f"  Test:  {len(test_pairs):,}")

# ── Save splits ───────────────────────────────────────────────
def save_split(pairs, name):
    with open(f"{CLEAN_DIR}/{name}.kp",
              'w', encoding='utf-8') as f:
        f.write('\n'.join([p[0] for p in pairs]))
    with open(f"{CLEAN_DIR}/{name}.hi",
              'w', encoding='utf-8') as f:
        f.write('\n'.join([p[1] for p in pairs]))

save_split(train_pairs, 'train')
save_split(dev_pairs,   'dev')
save_split(test_pairs,  'test')

print("\nSample pairs:")
for i in range(3):
    print(f"\n  K: {train_pairs[i][0][:65]}")
    print(f"  H: {train_pairs[i][1][:65]}")

# ── Download Devanagari font ──────────────────────────────────
print("\nDownloading font...")
subprocess.run(['wget', '-q', '-O', FONT_PATH,
    'https://github.com/googlefonts/noto-fonts/raw/'
    'main/hinted/ttf/NotoSansDevanagari/'
    'NotoSansDevanagari-Regular.ttf'])
print(f"Font: "
      f"{'✅' if os.path.exists(FONT_PATH) else '❌'}")

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✅ Cell 1 complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.6 MB/s eta 0:00:00
Kinnauri: 20,307
Hindi:    20,307
✅ Files perfectly aligned
After length filter: 20,304 pairs (skipped 3)

Split:
  Train: 19,304
  Dev:   500
  Test:  500

Sample pairs:

  K: दुईये ऐह मुलुक फिल्म होर केह काम खोटिह देह।
  H: दोनों ने कई फ़िल्मों में काम किया।

  K: भारौत हौयौ दुनिया आपदा कै आलाग सोच़, आलाग नीति, एकजुटता सि सात को
  H: भारत  इस विश्व आपदा में  अलग सोच, अलग नीति, एकजुटता के साथ कोरोना

  K: हाऊ खुश सुह किह तुह मालाह एसाह साएन सेह।
  H: मुझे खुशी है कि आप मेरी देखभाल कर रहे हैं।

Font: ✅
Device: cuda
✅ Cell 1 complete


In [2]:
print("=== Cell 2: Charset + CTC ===")

char_set = set()
with open(f"{CLEAN_DIR}/train.kp",
          encoding='utf-8') as f:
    for line in f:
        for ch in line.strip():
            char_set.add(ch)
char_set.discard(' ')

SPECIAL_TOKENS   = ['[PAD]', '[UNK]', ' ']
KIN_CHARS        = sorted(list(char_set))
ALL_CHARS        = SPECIAL_TOKENS + KIN_CHARS
NUM_CTC_CLASSES  = len(ALL_CHARS) + 1
BATCH_MAX_LENGTH = 30

print(f"Kinnauri chars:  {len(KIN_CHARS)}")
print(f"Total charset:   {len(ALL_CHARS)}")
print(f"CTC classes:     {NUM_CTC_CLASSES}")

class KinnauriCTCConverter:
    def __init__(self, characters):
        self.dict      = {c: i+1 for i, c
                          in enumerate(characters)}
        self.character = ['[CTCblank]'] + characters
        print(f"CTC vocab: {len(self.character)}")

    def encode(self, word_string,
               batch_max_length=30):
        length_list  = []
        index_tensor = torch.LongTensor(
            len(word_string),
            batch_max_length
        ).fill_(self.dict['[PAD]'])
        for i, word in enumerate(word_string):
            chars  = list(word)[:batch_max_length]
            length = len(chars)
            length_list.append(length)
            idxs   = [self.dict.get(
                          ch, self.dict['[UNK]'])
                      for ch in chars]
            if length > 0:
                index_tensor[i][:length] = \
                    torch.LongTensor(idxs)
        return (index_tensor.to(device),
                torch.IntTensor(
                    length_list).to(device))

    def decode(self, word_index, word_length):
        results = []
        for idx, length in enumerate(word_length):
            chars = []
            wi    = word_index[idx, :]
            for i in range(length):
                if (wi[i] != 0 and
                    not (i > 0 and
                         wi[i-1] == wi[i])):
                    chars.append(
                        self.character[wi[i]])
            results.append(''.join(chars))
        return results

converter = KinnauriCTCConverter(ALL_CHARS)

with open(f"{WORK_DIR}/kinnauri_charset.json",
          'w', encoding='utf-8') as f:
    json.dump(ALL_CHARS, f, ensure_ascii=False)

print("✅ Cell 2 complete")

=== Cell 2: Charset + CTC ===
Kinnauri chars:  127
Total charset:   130
CTC classes:     131
CTC vocab: 131
✅ Cell 2 complete


In [3]:
from torchvision import transforms
print("=== Cell 3: Scalogram Generation ===")

IMG_H=32; IMG_W=100; RENDER_H=64; RENDER_W=300
FONT_SIZE=40; WAVELET='morl'; NUM_SCALES=32
SCALES = np.geomspace(1, 32, num=NUM_SCALES)

try:
    font = ImageFont.truetype(FONT_PATH, FONT_SIZE)
    print("✅ Font loaded")
except Exception as e:
    print(f"⚠️ {e}")
    font = ImageFont.load_default()

def render_word(text):
    text = unicodedata.normalize('NFC', text.strip())
    img  = Image.new('L', (RENDER_W, RENDER_H),
                     color=0)
    draw = ImageDraw.Draw(img)
    try:
        bbox = draw.textbbox(
            (0,0), text, font=font)
        y    = max(0,
                   (RENDER_H-(bbox[3]-bbox[1]))//2)
        draw.text((4, y), text,
                  font=font, fill=255)
    except:
        draw.text((4, 10), text,
                  font=font, fill=255)
    return np.array(img, dtype=np.float32)

def apply_cwt(img_gray):
    scalogram = np.zeros(
        (NUM_SCALES, RENDER_W), dtype=np.float32)
    for row in img_gray:
        if row.max() < 1e-6:
            continue
        mu, sigma = row.mean(), row.std()
        signal    = (row - mu) / (sigma + 1e-8)
        coeffs, _ = pywt.cwt(
            signal, SCALES, WAVELET)
        scalogram += np.log1p(np.abs(coeffs))
    s_min = scalogram.min()
    s_max = scalogram.max()
    if s_max - s_min > 1e-8:
        scalogram = ((scalogram - s_min) /
                     (s_max - s_min) * 255.0)
    return scalogram.astype(np.uint8)

def word_to_scalogram(word):
    gray = render_word(word)
    scal = apply_cwt(gray)
    img  = Image.fromarray(scal, mode='L')
    return img.resize(
        (IMG_W, IMG_H), Image.LANCZOS)

# Collect unique Kinnauri words from all splits
all_words = set()
for split in ['train', 'dev', 'test']:
    with open(f"{CLEAN_DIR}/{split}.kp",
              encoding='utf-8') as f:
        for line in f:
            all_words.update(line.strip().split())

print(f"Unique Kinnauri words: {len(all_words):,}")

SCALO_DIR   = f"{WORK_DIR}/kinnauri_scalograms"
os.makedirs(SCALO_DIR, exist_ok=True)

word_list   = sorted(list(all_words))
word_to_idx = {w: i
               for i, w in enumerate(word_list)}

with open(f"{WORK_DIR}/kinnauri_word_to_idx.json",
          'w', encoding='utf-8') as f:
    json.dump(word_to_idx, f, ensure_ascii=False)

print(f"Generating {len(word_list):,} scalograms...")
errors = 0
t0     = time.time()

for i, word in enumerate(word_list):
    out_path = f"{SCALO_DIR}/{i:06d}.png"
    if os.path.exists(out_path):
        continue
    try:
        word_to_scalogram(word).save(out_path)
    except:
        errors += 1
    if (i+1) % 2000 == 0:
        elapsed = time.time() - t0
        rate    = (i+1) / elapsed
        eta     = (len(word_list)-i-1) / rate / 60
        print(f"  {i+1:5d}/{len(word_list)} | "
              f"{elapsed:.0f}s | "
              f"ETA:{eta:.1f}m | "
              f"Err:{errors}")

print(f"\n✅ Done: {len(word_list)-errors:,}")
print(f"   Errors: {errors}")
print("✅ Cell 3 complete")

=== Cell 3: Scalogram Generation ===
✅ Font loaded
Unique Kinnauri words: 27,627
Generating 27,627 scalograms...
   2000/27627 | 190s | ETA:40.6m | Err:0
   4000/27627 | 388s | ETA:38.2m | Err:0
   6000/27627 | 585s | ETA:35.1m | Err:0
   8000/27627 | 793s | ETA:32.4m | Err:0
  10000/27627 | 992s | ETA:29.1m | Err:0
  12000/27627 | 1197s | ETA:26.0m | Err:0
  14000/27627 | 1401s | ETA:22.7m | Err:0
  16000/27627 | 1598s | ETA:19.4m | Err:0
  18000/27627 | 1800s | ETA:16.0m | Err:0
  20000/27627 | 2000s | ETA:12.7m | Err:0
  22000/27627 | 2195s | ETA:9.4m | Err:0
  24000/27627 | 2394s | ETA:6.0m | Err:0
  26000/27627 | 2594s | ETA:2.7m | Err:0

✅ Done: 27,627
   Errors: 0
✅ Cell 3 complete


In [4]:
from torch.utils.data import Dataset, DataLoader
print("=== Cell 4: DataLoader ===")

eval_transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
train_transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.RandomApply(
        [transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

class KinnauriDataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.samples   = []
        with open(f"{CLEAN_DIR}/{split}.kp",
                  encoding='utf-8') as f:
            for line in f:
                for word in line.strip().split():
                    if word in word_to_idx:
                        idx = word_to_idx[word]
                        p   = (f"{SCALO_DIR}"
                               f"/{idx:06d}.png")
                        if os.path.exists(p):
                            self.samples.append(
                                (p, word))
        print(f"[{split}] "
              f"{len(self.samples):,} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, word = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, word

def ctc_collate(batch):
    imgs, texts = zip(*batch)
    return torch.stack(imgs, 0), list(texts)

BATCH_SIZE = 256

train_ds = KinnauriDataset('train', train_transform)
dev_ds   = KinnauriDataset('dev',   eval_transform)
test_ds  = KinnauriDataset('test',  eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True,  num_workers=2,
    pin_memory=True, collate_fn=ctc_collate)
dev_loader   = DataLoader(
    dev_ds,   batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=True, collate_fn=ctc_collate)
test_loader  = DataLoader(
    test_ds,  batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=True, collate_fn=ctc_collate)

print(f"\nBatch size:    {BATCH_SIZE}")
print(f"Train batches: {len(train_loader):,}")
print(f"Dev batches:   {len(dev_loader):,}")
print(f"Test batches:  {len(test_loader):,}")

imgs, texts = next(iter(train_loader))
print(f"\nBatch shape: {imgs.shape}")
print(f"Sample words: {texts[:5]}")
idx_t, len_t = converter.encode(
    texts[:3],
    batch_max_length=BATCH_MAX_LENGTH)
decoded = converter.decode(idx_t, len_t)
print(f"CTC roundtrip: {decoded[:3]}")
match = all(d==t
            for d,t in zip(decoded[:3], texts[:3]))
print(f"Match: {'✅' if match else '❌'}")
print("✅ Cell 4 complete")

=== Cell 4: DataLoader ===
[train] 156,280 samples
[dev] 4,032 samples
[test] 4,197 samples

Batch size:    256
Train batches: 611
Dev batches:   16
Test batches:  17

Batch shape: torch.Size([256, 1, 32, 100])
Sample words: ['तेले', 'रोहवे', 'एक', 'गान', 'कासी']
CTC roundtrip: ['तेले', 'रोहवे', 'एक']
Match: ✅
✅ Cell 4 complete


In [5]:
from modules.svtr import SVTR
print("=== Cell 5: SVTR+CTC Model ===")

class Stage1Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.FeatureExtraction = SVTR(
            img_size=[32, 100],
            in_channels=1,
            out_channels=256)
        self.AdaptiveAvgPool = \
            nn.AdaptiveAvgPool2d((None, 1))
        self.Prediction = nn.Linear(256, num_classes)

    def forward(self, x):
        v = self.FeatureExtraction(x)
        v = v.permute(0, 3, 1, 2)
        v = self.AdaptiveAvgPool(v)
        v = v.squeeze(3)
        return self.Prediction(v.contiguous())

model = Stage1Model(NUM_CTC_CLASSES).to(device)

for name, param in model.named_parameters():
    try:
        if 'bias'    in name:
            nn.init.constant_(param, 0.0)
        elif 'weight' in name:
            nn.init.kaiming_normal_(param)
    except:
        if 'weight' in name:
            param.data.fill_(1)

total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

model.eval()
with torch.no_grad():
    dummy = torch.randn(4, 1, 32, 100).to(device)
    out   = model(dummy)
    print(f"Input:  {dummy.shape}")
    print(f"Output: {out.shape}")
    t_ok = out.shape[1] >= BATCH_MAX_LENGTH
    print(f"T={out.shape[1]} >= "
          f"BATCH_MAX_LENGTH={BATCH_MAX_LENGTH}: "
          f"{'✅' if t_ok else '❌'}")

ctc_loss_fn = nn.CTCLoss(
    zero_infinity=True).to(device)
print("✅ Cell 5 complete")

=== Cell 5: SVTR+CTC Model ===
Parameters: 4,997,891
Input:  torch.Size([4, 1, 32, 100])
Output: torch.Size([4, 25, 131])
T=25 >= BATCH_MAX_LENGTH=30: ❌
✅ Cell 5 complete


In [6]:
from torch.optim import AdamW
print("=== Cell 6: Training ===")

EPOCHS    = 20
LR        = 1e-3
PATIENCE  = 5
CKPT_PATH = f"{WORK_DIR}/kinnauri_stage1_best.pt"

optimizer   = AdamW(model.parameters(),
                    lr=LR, weight_decay=1e-4)
total_steps = len(train_loader) * EPOCHS
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    total_steps=total_steps,
    div_factor=20,
    final_div_factor=1000,
    pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()

def compute_cer(preds, targets):
    from nltk.metrics.distance import edit_distance
    td, tl = 0, 0
    for p, t in zip(preds, targets):
        td += edit_distance(p, t)
        tl += max(len(t), 1)
    return td / tl

def run_train_epoch(ep):
    model.train()
    total, nb   = 0, len(train_loader)
    running, t0 = 0, time.time()
    for i, (imgs, texts) in enumerate(train_loader):
        imgs = imgs.to(device)
        labels, label_lengths = converter.encode(
            texts,
            batch_max_length=BATCH_MAX_LENGTH)
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda',
                            dtype=torch.float16):
            preds = model(imgs)
            plog  = preds.log_softmax(2)\
                         .permute(1,0,2)
            psz   = torch.IntTensor(
                [preds.size(1)] * imgs.size(0)
            ).to(device)
            loss  = ctc_loss_fn(
                plog, labels, psz, label_lengths)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total   += loss.item()
        running += loss.item()
        if (i+1) % 50 == 0:
            eta = ((time.time()-t0) / (i+1)
                   * (nb-i-1) / 60)
            print(f"  Ep{ep+1} "
                  f"B{i+1:4d}/{nb} "
                  f"Loss:{running/50:.4f} "
                  f"ETA:{eta:.1f}m")
            running = 0
    return total / nb

def run_eval(loader):
    model.eval()
    total, nb          = 0, 0
    all_preds, all_tgt = [], []
    with torch.no_grad():
        for imgs, texts in loader:
            imgs = imgs.to(device)
            labels, label_lengths = converter.encode(
                texts,
                batch_max_length=BATCH_MAX_LENGTH)
            with torch.autocast(device_type='cuda',
                                dtype=torch.float16):
                preds = model(imgs)
                plog  = preds.log_softmax(2)\
                             .permute(1,0,2)
                psz   = torch.IntTensor(
                    [preds.size(1)] * imgs.size(0)
                ).to(device)
                loss  = ctc_loss_fn(
                    plog, labels, psz, label_lengths)
            total += loss.item()
            nb    += 1
            _, pi  = preds.max(2)
            dec    = converter.decode(pi, psz)
            all_preds.extend(dec)
            all_tgt.extend(texts)
    cer = compute_cer(all_preds, all_tgt)
    return (total/max(nb,1), cer,
            all_preds[:5], all_tgt[:5])

# ── Training loop ─────────────────────────────────────────────
best_cer, patience_cnt = float('inf'), 0
history = {
    'train_loss':[], 'dev_loss':[], 'dev_cer':[]}

print(f"{'='*55}")
print(f"  Kinnauri SVTR+CTC Training")
print(f"  Epochs:{EPOCHS} | "
      f"Batches:{len(train_loader)} | "
      f"Classes:{NUM_CTC_CLASSES}")
print(f"{'='*55}")

for epoch in range(EPOCHS):
    t0 = time.time()
    print(f"\n── Epoch {epoch+1}/{EPOCHS} ──")

    tr_loss = run_train_epoch(epoch)
    dv_loss, dv_cer, sp, st = run_eval(dev_loader)

    history['train_loss'].append(tr_loss)
    history['dev_loss'].append(dv_loss)
    history['dev_cer'].append(dv_cer)

    better = dv_cer < best_cer
    print(f"\n  Train Loss: {tr_loss:.4f}")
    print(f"  Dev Loss:   {dv_loss:.4f}")
    print(f"  Dev CER:    {dv_cer:.4f} "
          f"({'✅' if better else '❌'})")
    print(f"  Best CER:   {best_cer:.4f} | "
          f"Time:{time.time()-t0:.0f}s")
    print(f"\n  Samples:")
    for p, t in zip(sp, st):
        print(f"    {'✅' if p==t else '❌'} "
              f"'{p}' | '{t}'")

    if better:
        best_cer       = dv_cer
        patience_cnt   = 0
        torch.save({
            'epoch':         epoch+1,
            'model_state':   model.state_dict(),
            'best_cer':      best_cer,
            'num_classes':   NUM_CTC_CLASSES,
            'all_chars':     ALL_CHARS,
            'word_to_idx':   word_to_idx,
            'batch_max_len': BATCH_MAX_LENGTH,
        }, CKPT_PATH)
        print(f"\n  ✅ Saved "
              f"CER:{best_cer:.4f} "
              f"Acc:{(1-best_cer)*100:.2f}%")
    else:
        patience_cnt += 1
        print(f"\n  Patience: "
              f"{patience_cnt}/{PATIENCE}")
        if patience_cnt >= PATIENCE:
            print("  Early stopping.")
            break

    with open(
            f"{WORK_DIR}/kinnauri_history.json",
            'w') as f:
        json.dump(history, f)

print(f"\n{'='*55}")
print(f"Done | CER:{best_cer:.4f} | "
      f"Acc:{(1-best_cer)*100:.2f}%")

=== Cell 6: Training ===
  Kinnauri SVTR+CTC Training
  Epochs:20 | Batches:611 | Classes:131

── Epoch 1/20 ──
  Ep1 B  50/611 Loss:7.7415 ETA:1.7m
  Ep1 B 100/611 Loss:3.7961 ETA:1.4m
  Ep1 B 150/611 Loss:3.1468 ETA:1.3m
  Ep1 B 200/611 Loss:2.5041 ETA:1.1m
  Ep1 B 250/611 Loss:2.0759 ETA:1.0m
  Ep1 B 300/611 Loss:1.7702 ETA:0.8m
  Ep1 B 350/611 Loss:1.5241 ETA:0.7m
  Ep1 B 400/611 Loss:1.3238 ETA:0.6m
  Ep1 B 450/611 Loss:1.1845 ETA:0.4m
  Ep1 B 500/611 Loss:1.0233 ETA:0.3m
  Ep1 B 550/611 Loss:0.9013 ETA:0.2m
  Ep1 B 600/611 Loss:0.8354 ETA:0.0m

  Train Loss: 2.2914
  Dev Loss:   0.6101
  Dev CER:    0.1845 (✅)
  Best CER:   inf | Time:104s

  Samples:
    ✅ 'जिह' | 'जिह'
    ✅ 'मोन' | 'मोन'
    ✅ 'लाह' | 'लाह'
    ✅ 'शाह' | 'शाह'
    ✅ 'सोमजिह' | 'सोमजिह'

  ✅ Saved CER:0.1845 Acc:81.55%

── Epoch 2/20 ──
  Ep2 B  50/611 Loss:0.7063 ETA:1.6m
  Ep2 B 100/611 Loss:0.6483 ETA:1.4m
  Ep2 B 150/611 Loss:0.5876 ETA:1.3m
  Ep2 B 200/611 Loss:0.5350 ETA:1.1m
  Ep2 B 250/611 Loss:0.4719 E

In [7]:
from transformers import (AutoModelForSeq2SeqLM,
                           AutoTokenizer)
print("=== Cell 7: NLLB + Hindi LM ===")

MODEL_NAME = "facebook/nllb-200-distilled-600M"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
nllb      = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME).to(device)
nllb.eval()
print("✅ NLLB-200 loaded")

# Kinnauri → Hindi
# Kinnauri Pahari uses Devanagari script
# No dedicated NLLB code for Kinnauri
# Using hin_Deva as closest linguistic proxy
SRC_LANG = "hin_Deva"
TGT_LANG = "hin_Deva"

def translate(sentence, num_beams=4):
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        sentence, return_tensors="pt",
        padding=True, truncation=True,
        max_length=256).to(device)
    tgt_id = tokenizer.convert_tokens_to_ids(
        TGT_LANG)
    with torch.no_grad():
        out = nllb.generate(
            **inputs,
            forced_bos_token_id=tgt_id,
            num_beams=num_beams,
            max_length=256,
            early_stopping=True)
    return tokenizer.decode(
        out[0], skip_special_tokens=True)

def beam_candidates(sentence,
                    num_beams=5, num_return=3):
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        sentence, return_tensors="pt",
        padding=True, truncation=True,
        max_length=256).to(device)
    tgt_id = tokenizer.convert_tokens_to_ids(
        TGT_LANG)
    with torch.no_grad():
        outputs = nllb.generate(
            **inputs,
            forced_bos_token_id=tgt_id,
            num_beams=num_beams,
            num_return_sequences=num_return,
            max_length=256,
            early_stopping=True)
    return [tokenizer.decode(
                o, skip_special_tokens=True)
            for o in outputs]

# Test
print("\nTranslation test:")
for k, h in train_pairs[:3]:
    out = translate(k)
    print(f"  KP:  {k[:60]}")
    print(f"  Ref: {h[:60]}")
    print(f"  NMT: {out[:60]}")
    print()

# ── Hindi Bigram LM ───────────────────────────────────────────
print("Building Hindi LM...")
hi_uni = Counter()
hi_bi  = defaultdict(Counter)

with open(f"{CLEAN_DIR}/train.hi",
          encoding='utf-8') as f:
    for line in f:
        words = (['<s>']
                 + line.strip().split()
                 + ['</s>'])
        for i, w in enumerate(words):
            hi_uni[w] += 1
            if i > 0:
                hi_bi[words[i-1]][w] += 1

hi_vocab = len(hi_uni)
print(f"Hindi vocab:   {hi_vocab:,}")
print(f"Hindi bigrams: {len(hi_bi):,}")

def lm_score(sentence, alpha=0.1):
    words = (['<s>']
             + sentence.strip().split()
             + ['</s>'])
    log_p = 0.0
    for i in range(1, len(words)):
        prev = words[i-1]
        curr = words[i]
        cp   = hi_uni.get(prev, 0)
        cb   = hi_bi[prev].get(curr, 0)
        prob = ((cb + alpha) /
                (cp + alpha * hi_vocab + 1e-10))
        log_p += math.log(max(prob, 1e-10))
    return log_p

print("✅ Cell 7 complete")

=== Cell 7: NLLB + Hindi LM ===
Loading facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB-200 loaded

Translation test:
  KP:  दुईये ऐह मुलुक फिल्म होर केह काम खोटिह देह।
  Ref: दोनों ने कई फ़िल्मों में काम किया।
  NMT: दो ऐह देश फिल्म होर केह काम खोटिह देह।

  KP:  भारौत हौयौ दुनिया आपदा कै आलाग सोच़, आलाग नीति, एकजुटता सि स
  Ref: भारत  इस विश्व आपदा में  अलग सोच, अलग नीति, एकजुटता के साथ क
  NMT: आपदाओं से निपटने के लिए एकजुटता, एकजुटता, एकजुटता, एकजुटता न

  KP:  हाऊ खुश सुह किह तुह मालाह एसाह साएन सेह।
  Ref: मुझे खुशी है कि आप मेरी देखभाल कर रहे हैं।
  NMT: हाउ खुश सुह किह तुह मालाह एसाह सायन सेह।

Building Hindi LM...
Hindi vocab:   18,827
Hindi bigrams: 18,826
✅ Cell 7 complete


In [8]:
print("=== Cell 8: Inference Pipeline ===")

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"✅ Loaded — "
      f"CER:{ckpt['best_cer']:.4f} "
      f"Acc:{(1-ckpt['best_cer'])*100:.2f}%")

def read_word(word):
    try:
        gray   = render_word(word)
        scal   = apply_cwt(gray)
        img    = Image.fromarray(scal, mode='L')
        img    = img.resize(
            (IMG_W, IMG_H), Image.LANCZOS)
        tensor = eval_transform(img)\
                     .unsqueeze(0).to(device)
        with torch.no_grad():
            p     = model(tensor)
            _, pi = p.max(2)
            ps    = torch.IntTensor(
                [p.size(1)]).to(device)
            return converter.decode(pi, ps)[0]
    except:
        return word

def read_word_noisy(word, noise_level=0.05):
    try:
        gray  = render_word(word)
        scal  = apply_cwt(gray).astype(np.float32)
        noise = np.random.normal(
            0, noise_level*255,
            scal.shape).astype(np.float32)
        noisy = np.clip(
            scal+noise, 0, 255).astype(np.uint8)
        img   = Image.fromarray(noisy, mode='L')
        img   = img.resize(
            (IMG_W, IMG_H), Image.LANCZOS)
        tensor= eval_transform(img)\
                    .unsqueeze(0).to(device)
        with torch.no_grad():
            p     = model(tensor)
            _, pi = p.max(2)
            ps    = torch.IntTensor(
                [p.size(1)]).to(device)
            return converter.decode(pi, ps)[0]
    except:
        return word

def full_pipeline(kin_sent):
    words  = kin_sent.strip().split()
    recog  = [read_word(w) or w for w in words]
    joined = ' '.join(recog)
    cands  = beam_candidates(joined)
    best   = sorted(cands,
                    key=lm_score,
                    reverse=True)[0]
    return best, joined, cands

def full_pipeline_noisy(kin_sent, noise=0.05):
    words  = kin_sent.strip().split()
    recog  = [read_word_noisy(w, noise) or w
              for w in words]
    joined = ' '.join(recog)
    return translate(joined), joined

# Quick test
print("\nPipeline test:")
for k, h in test_pairs[:3]:
    best, recog, cands = full_pipeline(k)
    print(f"  KP:   {k[:60]}")
    print(f"  Ref:  {h[:60]}")
    print(f"  OCR:  {recog[:60]}")
    print(f"  Best: {best[:60]}")
    print()

print("✅ Cell 8 complete")

=== Cell 8: Inference Pipeline ===
✅ Loaded — CER:0.0060 Acc:99.40%

Pipeline test:
  KP:   ऐक्या शोरोह-शोशाई होयो थाकदेस ज़िह आफरोह बोवारी सिह छ़ेल्डी 
  Ref:  कितने सास ससुर ऐसे हैं जो अपनी बहू और बेटी में कोई अंतर नहीं
  OCR:  ऐक्या शोरोह-शोशाई होयो थाकदेस ज़िह आफरोह बोवारी सिह छ़ेल्डी 
  Best: क्या शोरोह-शोशाई हो थाकदेस ज़िह अफरोह बोवारी सिह छल्डी सेह क

  KP:   होयो देखिह देह कुन्ती आमां लाह मुलुक ङार आओस
  Ref:  यह देखकर माता कुन्ती को और अधिक क्रोध आ गया
  OCR:  होयो देखिह देह कुन्ती आमां लाह मुलुक ङार आओस
  Best: होहो देखो देह कुंती माँ लाह देश गार आआस

  KP:   स्कुल रोह कॉम्प्लैक्स केह एकह मुलुक बोडोह सोर थियोह।
  Ref:  स्कूल के कॉम्प्लैक्स में एक बहुत बड़ा तालाब, था।
  OCR:  स्कुल रोह कॉमलैक्स केह एकह मुलुक बोडोह सोर थियोह।
  Best: स्कूल रोह कॉमलैक्स केह एकह देश बोडोह सोर था।

✅ Cell 8 complete


In [9]:
import sacrebleu
import evaluate as hf_evaluate
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import (PatternFill, Font,
                              Alignment)
from openpyxl.utils import get_column_letter

print("=== Cell 9: Evaluation + Excel ===")

TEST_KP   = [p[0] for p in test_pairs]
TEST_HI   = [p[1] for p in test_pairs]
TEST_N    = len(TEST_KP)
NOISE_LVL = 0.05

print(f"Evaluating {TEST_N} sentences...")

results     = []
preds_nolm  = []
preds_lm    = []
preds_noisy = []
refs        = []
t0          = time.time()

for i, (ks, ref) in enumerate(
        zip(TEST_KP, TEST_HI)):
    try:
        # No LM
        words  = ks.split()
        recog  = [read_word(w) or w for w in words]
        joined = ' '.join(recog)
        nolm   = translate(joined)

        # With LM
        lm_out, _, cands = full_pipeline(ks)

        # Noisy
        noisy_tr, noisy_ocr = \
            full_pipeline_noisy(ks, NOISE_LVL)

        ocr_ok = ('✅'
                  if joined.strip()==ks.strip()
                  else '⚠️')
        lm_chg = ('✅' if lm_out != nolm
                  else '➖')

        results.append({
            'Sr.No':              i+1,
            'Kinnauri_Input':     ks,
            'Reference_Hindi':    ref,
            'OCR_Output':         joined,
            'OCR_Match':          ocr_ok,
            'Translation_NoLM':   nolm,
            'Translation_WithLM': lm_out,
            'LM_Candidates':      ' | '.join(cands),
            'LM_Changed':         lm_chg,
            'OCR_Noisy':          noisy_ocr,
            'Translation_Noisy':  noisy_tr,
        })
        preds_nolm.append(nolm)
        preds_lm.append(lm_out)
        preds_noisy.append(noisy_tr)
        refs.append(ref)

    except Exception as e:
        results.append({
            'Sr.No':i+1,
            'Kinnauri_Input':ks,
            'Reference_Hindi':ref,
            'OCR_Output':'','OCR_Match':'❌',
            'Translation_NoLM':'',
            'Translation_WithLM':'',
            'LM_Candidates':'','LM_Changed':'❌',
            'OCR_Noisy':'','Translation_Noisy':'',
        })
        preds_nolm.append('')
        preds_lm.append('')
        preds_noisy.append('')
        refs.append(ref)

    if (i+1) % 50 == 0:
        print(f"  {i+1}/{TEST_N} | "
              f"{time.time()-t0:.0f}s")

df      = pd.DataFrame(results)
ocr_acc = (sum(1 for r in df['OCR_Match']
               if r=='✅') / len(df) * 100)

def get_metrics(preds, refs, label):
    valid = [(p,r) for p,r in zip(preds,refs)
             if str(p).strip() and str(r).strip()]
    if not valid:
        return {'BLEU':0,'TER':100,
                'METEOR':0,'NIST':0}
    pl = [x[0] for x in valid]
    rl = [x[1] for x in valid]
    bleu = sacrebleu.corpus_bleu(pl,[rl]).score
    ter  = sacrebleu.corpus_ter(pl,[rl]).score
    nist = sacrebleu.corpus_bleu(
        pl,[rl],smooth_method='add-k').score
    me   = hf_evaluate.load('meteor')
    met  = me.compute(
        predictions=pl,
        references=rl)['meteor'] * 100
    print(f"  [{label}] "
          f"BLEU:{bleu:.2f} TER:{ter:.2f} "
          f"METEOR:{met:.2f} NIST:{nist:.2f}")
    return {'BLEU':bleu,'TER':ter,
            'METEOR':met,'NIST':nist}

print("\nComputing metrics...")
m_nolm  = get_metrics(preds_nolm, refs,"No LM")
m_lm    = get_metrics(preds_lm,   refs,"With LM")
m_noisy = get_metrics(preds_noisy,refs,"Noisy")

# ── Excel ─────────────────────────────────────────────────────
out_path = f"{WORK_DIR}/kinnauri_hindi_results.xlsx"

def style_ws(ws, hc="1F4E79"):
    hf  = PatternFill(start_color=hc,
                      end_color=hc,
                      fill_type="solid")
    hft = Font(color="FFFFFF", bold=True,
               size=11)
    for cell in ws[1]:
        cell.fill = hf
        cell.font = hft
        cell.alignment = Alignment(
            horizontal='center',
            wrap_text=True)
    ws.row_dimensions[1].height = 30
    for i, row in enumerate(
            ws.iter_rows(
                min_row=2,
                max_row=ws.max_row)):
        c = "EBF3FB" if i%2==0 else "FFFFFF"
        for cell in row:
            cell.fill = PatternFill(
                start_color=c,
                end_color=c,
                fill_type="solid")
            cell.alignment = Alignment(
                wrap_text=True,
                vertical='top')
    for col in ws.columns:
        cl = get_column_letter(col[0].column)
        ml = max(
            (len(str(c.value))
             for c in col if c.value),
            default=10)
        ws.column_dimensions[cl].width = \
            min(ml+4, 55)
    ws.freeze_panes = 'A2'

with pd.ExcelWriter(out_path,
                    engine='openpyxl') as w:

    # Sheet 1 — Mentor format
    pd.DataFrame({
        'Language_Pair':
            ['Kinnauri-Hindi (kp-hi)']*4,
        'Metric':
            ['BLEU-4','TER','METEOR','NIST'],
        'No_LM':
            [f"{m_nolm['BLEU']:.2f}",
             f"{m_nolm['TER']:.2f}",
             f"{m_nolm['METEOR']:.2f}",
             f"{m_nolm['NIST']:.2f}"],
        'With_LM':
            [f"{m_lm['BLEU']:.2f}",
             f"{m_lm['TER']:.2f}",
             f"{m_lm['METEOR']:.2f}",
             f"{m_lm['NIST']:.2f}"],
        'No_Noise':
            [f"{m_nolm['BLEU']:.2f}",
             f"{m_nolm['TER']:.2f}",
             f"{m_nolm['METEOR']:.2f}",
             f"{m_nolm['NIST']:.2f}"],
        'With_5pct_Noise':
            [f"{m_noisy['BLEU']:.2f}",
             f"{m_noisy['TER']:.2f}",
             f"{m_noisy['METEOR']:.2f}",
             f"{m_noisy['NIST']:.2f}"],
        'Better_When':
            ['Higher↑','Lower↓',
             'Higher↑','Higher↑'],
        'OCR_Accuracy':
            [f"{ocr_acc:.2f}%",'-','-','-'],
        'Test_Sentences':
            [str(TEST_N),'-','-','-'],
    }).to_excel(w,
        sheet_name='Metrics (Mentor Format)',
        index=False)

    # Sheet 2 — All results
    df.to_excel(w,
        sheet_name='All Results',
        index=False)

    # Sheet 3 — Sample 10
    df[['Sr.No','Kinnauri_Input',
        'Reference_Hindi','OCR_Output',
        'OCR_Match','Translation_NoLM',
        'Translation_WithLM','LM_Changed',
        'Translation_Noisy']
      ].head(10).to_excel(w,
        sheet_name='Sample 10',
        index=False)

    # Sheet 4 — LM Improved
    df[df['LM_Changed']=='✅'].to_excel(w,
        sheet_name='LM Improved',
        index=False)

    # Sheet 5 — OCR Errors
    df[df['OCR_Match']!='✅'].to_excel(w,
        sheet_name='OCR Errors',
        index=False)

    # Sheet 6 — System Info
    pd.DataFrame({
        'Component': [
            'Language Pair',
            'Source Language',
            'Target Language',
            'Script',
            'Total Pairs',
            'Train / Dev / Test',
            'Random Seed',
            'OCR Architecture',
            'OCR Best CER',
            'OCR Word Accuracy',
            'Image Representation',
            'Wavelet',
            'Image Size',
            'Unique Kinnauri Words',
            'Translation Model',
            'NLLB Source Code',
            'NLLB Target Code',
            'Language Model',
            'LM Training Data',
            'LM Hindi Vocabulary',
            'LM Position',
            'Noise Type',
            'Noise Level',
            'Note',
            'Pipeline',
        ],
        'Details': [
            'Kinnauri Pahari → Hindi',
            'Kinnauri Pahari '
            '(endangered Tibeto-Burman, '
            'Devanagari script)',
            'Standard Hindi',
            'Devanagari',
            str(len(pairs)),
            f"{len(train_pairs):,} / "
            f"{len(dev_pairs):,} / "
            f"{len(test_pairs):,}  (seed=42)",
            '42',
            'SVTR + CTC '
            '(CLL-STR, ICASSP 2024)',
            f"{best_cer:.4f}",
            f"{(1-best_cer)*100:.2f}%",
            'CWT Scalogram '
            '(Morlet wavelet)',
            'Morlet (morl), '
            '32 log-spaced scales (1→32)',
            '32 × 100 pixels (grayscale)',
            f"{len(word_list):,}",
            'Facebook NLLB-200-distilled-600M',
            'hin_Deva',
            'hin_Deva',
            'Bigram N-Gram LM '
            '(Hindi target-side)',
            f'Kinnauri-Hindi training set '
            f'({len(train_pairs):,})',
            f"{hi_vocab:,} Hindi words",
            'Target-side reranking '
            '(3 beam candidates)',
            'Gaussian noise on '
            'scalogram pixels',
            f"{NOISE_LVL*100:.0f}%",
            'Kinnauri Pahari is an '
            'endangered Tibeto-Burman '
            'language; NLLB has no '
            'dedicated code so hin_Deva '
            'used as Devanagari proxy',
            'Text→CWT Scalogram'
            '→SVTR+CTC→Join'
            '→NLLB-200→Hindi LM→Hindi',
        ]
    }).to_excel(w,
        sheet_name='System Info',
        index=False)

# Apply styling
wb = load_workbook(out_path)
colors = {
    'Metrics (Mentor Format)': '1F4E79',
    'All Results':             '145A32',
    'Sample 10':               '6C3483',
    'LM Improved':             '7D6608',
    'OCR Errors':              '78281F',
    'System Info':             '1A5276',
}
for sn in wb.sheetnames:
    style_ws(wb[sn], colors.get(sn,'1F4E79'))
wb.save(out_path)

# ── Final summary ─────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  KINNAURI-HINDI FINAL RESULTS")
print(f"{'='*55}")
print(f"  Language: Kinnauri Pahari → Hindi")
print(f"  Test:     {TEST_N} sentences")
print(f"  OCR CER:  {best_cer:.4f}")
print(f"  OCR Acc:  {ocr_acc:.2f}%")
print(f"\n  {'Metric':<10} {'NoLM':>7} "
      f"{'LM':>7} {'Noisy':>7}")
print(f"  {'-'*34}")
for m in ['BLEU','TER','METEOR','NIST']:
    print(f"  {m:<10} "
          f"{m_nolm[m]:>7.2f} "
          f"{m_lm[m]:>7.2f} "
          f"{m_noisy[m]:>7.2f}")
print(f"{'='*55}")
print(f"\n✅ Saved → {out_path}")
print(f"\nFill Google Sheet from:")
print(f"  Sheet: 'Metrics (Mentor Format)'")

=== Cell 9: Evaluation + Excel ===
Evaluating 500 sentences...
  50/500 | 205s
  100/500 | 417s
  150/500 | 645s
  200/500 | 850s
  250/500 | 1043s
  300/500 | 1252s
  350/500 | 1467s
  400/500 | 1678s
  450/500 | 1900s
  500/500 | 2146s

Computing metrics...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


  [No LM] BLEU:2.90 TER:101.89 METEOR:11.13 NIST:2.94


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  [With LM] BLEU:3.11 TER:95.48 METEOR:11.19 NIST:3.16


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  [Noisy] BLEU:0.91 TER:102.58 METEOR:4.91 NIST:0.97

  KINNAURI-HINDI FINAL RESULTS
  Language: Kinnauri Pahari → Hindi
  Test:     500 sentences
  OCR CER:  0.0060
  OCR Acc:  88.80%

  Metric        NoLM      LM   Noisy
  ----------------------------------
  BLEU          2.90    3.11    0.91
  TER         101.89   95.48  102.58
  METEOR       11.13   11.19    4.91
  NIST          2.94    3.16    0.97

✅ Saved → /kaggle/working/kinnauri_hindi_results.xlsx

Fill Google Sheet from:
  Sheet: 'Metrics (Mentor Format)'
